# Part 4 – LLM Powered Feature: Model Prediction Explanation Pipeline

This notebook demonstrates an LLM-powered feature built on top of the machine learning model developed in Part 3.

## 1. Import Required Libraries

In [2]:
import os
import json
import re
import requests
import joblib

import pandas as pd
import numpy as np

from jsonschema import validate, ValidationError

## 2. Load the Best Model

In [3]:
best_pipeline = joblib.load("../output/best_model.pkl")

print("Best model loaded successfully!")

Best model loaded successfully!


## 3. Load API Key

In [7]:
from dotenv import load_dotenv
import os

# Load the .env file
load_dotenv("../.env")

# Check if the key exists
print("LLM_API_KEY exists:", "LLM_API_KEY" in os.environ)

# Read the key
api_key = os.getenv("LLM_API_KEY")

if api_key:
    print("API Key Loaded Successfully!")
    print("First 10 characters:", api_key[:10] + "...")
else:
    print("API Key Not Found!")

LLM_API_KEY exists: True
API Key Loaded Successfully!
First 10 characters: sk-or-v1-5...


## 4. Create Reusable LLM Function

In [34]:
# OpenRouter API URL
url = "https://openrouter.ai/api/v1/chat/completions"


def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "openrouter/free",
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    if response.status_code != 200:
        print("Status Code:", response.status_code)
        print(response.text)
        return None

    return response.json()["choices"][0]["message"]["content"]

## 5. Test the LLM Connection

In [35]:
system_prompt = "You are a helpful assistant."

user_prompt = "Reply with only the word: hello"

response = call_llm(
    system_prompt,
    user_prompt
)

print("LLM Response")
print("----------------")
print(response)

LLM Response
----------------


hello



## 6. Define JSON Output Schema

In [36]:
schema = {
    "type": "object",
    "properties": {
        "prediction_label": {
            "type": "string"
        },
        "confidence_level": {
            "type": "string"
        },
        "top_reason": {
            "type": "string"
        },
        "second_reason": {
            "type": "string"
        },
        "next_step": {
            "type": "string"
        }
    },
    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ]
}

## 7. Personal Information Guardrail

In [13]:
def has_pii(text):

    email_pattern = r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+"

    phone_pattern = r"\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b"

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )

## 8. Test the Guardrail

In [14]:
blocked_input = "My email is student@gmail.com"

clean_input = "The student scored 15 marks."


print("Blocked Input")
print("----------------")
print(has_pii(blocked_input))


print()

print("Clean Input")
print("----------------")
print(has_pii(clean_input))

Blocked Input
----------------
True

Clean Input
----------------
False


## 9. Create Sample Student Records

In [15]:
# Three sample students

student_1 = {
    "school": "GP",
    "sex": "F",
    "age": 17,
    "address": "U",
    "famsize": "GT3",
    "Pstatus": "T",
    "Medu": 4,
    "Fedu": 4,
    "Mjob": "teacher",
    "Fjob": "teacher",
    "reason": "course",
    "guardian": "mother",
    "traveltime": 1,
    "studytime": 4,
    "failures": 0,
    "schoolsup": "no",
    "famsup": "yes",
    "paid": "no",
    "activities": "yes",
    "nursery": "yes",
    "higher": "yes",
    "internet": "yes",
    "romantic": "no",
    "famrel": 5,
    "freetime": 3,
    "goout": 2,
    "Dalc": 1,
    "Walc": 1,
    "health": 5,
    "absences": 2,
    "G1": 15,
    "G2": 16
}

student_2 = {
    "school": "MS",
    "sex": "M",
    "age": 18,
    "address": "R",
    "famsize": "LE3",
    "Pstatus": "A",
    "Medu": 1,
    "Fedu": 1,
    "Mjob": "other",
    "Fjob": "other",
    "reason": "other",
    "guardian": "other",
    "traveltime": 4,
    "studytime": 1,
    "failures": 3,
    "schoolsup": "yes",
    "famsup": "no",
    "paid": "yes",
    "activities": "no",
    "nursery": "no",
    "higher": "no",
    "internet": "no",
    "romantic": "yes",
    "famrel": 2,
    "freetime": 5,
    "goout": 5,
    "Dalc": 4,
    "Walc": 5,
    "health": 2,
    "absences": 18,
    "G1": 8,
    "G2": 7
}

student_3 = {
    "school": "GP",
    "sex": "F",
    "age": 16,
    "address": "U",
    "famsize": "GT3",
    "Pstatus": "T",
    "Medu": 3,
    "Fedu": 2,
    "Mjob": "health",
    "Fjob": "services",
    "reason": "home",
    "guardian": "mother",
    "traveltime": 2,
    "studytime": 3,
    "failures": 1,
    "schoolsup": "no",
    "famsup": "yes",
    "paid": "no",
    "activities": "yes",
    "nursery": "yes",
    "higher": "yes",
    "internet": "yes",
    "romantic": "no",
    "famrel": 4,
    "freetime": 3,
    "goout": 3,
    "Dalc": 1,
    "Walc": 2,
    "health": 4,
    "absences": 5,
    "G1": 12,
    "G2": 13
}

students = [student_1, student_2, student_3]

print("Three sample students created successfully!")

Three sample students created successfully!


## 10A. Encode Student Record

In [17]:
# Feature names used to train the model

training_columns = [
    'age', 'Medu', 'Fedu', 'traveltime', 'studytime',
    'failures', 'famrel', 'freetime', 'goout', 'Dalc',
    'Walc', 'health', 'absences', 'G1', 'G2',
    'school_MS', 'sex_M', 'address_U', 'famsize_LE3',
    'Pstatus_T', 'Mjob_health', 'Mjob_other',
    'Mjob_services', 'Mjob_teacher', 'Fjob_health',
    'Fjob_other', 'Fjob_services', 'Fjob_teacher',
    'reason_home', 'reason_other', 'reason_reputation',
    'guardian_mother', 'guardian_other',
    'schoolsup_yes', 'famsup_yes', 'paid_yes',
    'activities_yes', 'nursery_yes', 'higher_yes',
    'internet_yes', 'romantic_yes'
]


def encode_record(student):

    df_student = pd.DataFrame([student])

    df_student = pd.get_dummies(df_student, drop_first=True)

    for col in training_columns:
        if col not in df_student.columns:
            df_student[col] = 0

    df_student = df_student[training_columns]

    return df_student

## 10. Generate Model Predictions

In [19]:
for i, student in enumerate(students, start=1):

    encoded_student = encode_record(student)

    prediction = best_pipeline.predict(encoded_student)[0]

    probability = best_pipeline.predict_proba(encoded_student)[0].max()

    print(f"Student {i}")
    print("----------------------------")
    print("Prediction :", prediction)
    print(f"Probability : {probability:.3f}")
    print()

Student 1
----------------------------
Prediction : 1
Probability : 0.901

Student 2
----------------------------
Prediction : 0
Probability : 0.739

Student 3
----------------------------
Prediction : 1
Probability : 0.861



## 11. Create the LLM Prompt

In [37]:
system_prompt = """
You are an educational AI assistant.

Your task is to explain a machine learning prediction.

Return ONLY valid JSON.

Do not include:
- User Safety
- Markdown
- Code blocks
- Explanations
- Any extra text

The JSON must contain exactly these fields:

{
  "prediction_label": "Pass or Fail",
  "confidence_level": "Low, Medium or High",
  "top_reason": "string",
  "second_reason": "string",
  "next_step": "string"
}
"""

## 12. Generate Prediction Explanation

In [38]:
for i, student in enumerate(students, start=1):

    encoded_student = encode_record(student)

    prediction = int(best_pipeline.predict(encoded_student)[0])

    probability = float(best_pipeline.predict_proba(encoded_student)[0].max())

    user_prompt = f"""
Student Features:

{json.dumps(student, indent=2)}

Prediction:
{prediction}

Prediction Probability:
{probability:.3f}

Explain this prediction.

Return ONLY valid JSON in this format:

{{
"prediction_label":"",
"confidence_level":"",
"top_reason":"",
"second_reason":"",
"next_step":""
}}
"""

    response = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    print(f"\nStudent {i}")
    print("-----------------------------")
    print(response)


Student 1
-----------------------------

{
  "prediction_label": "Pass",
  "confidence_level": "High",
  "top_reason": "Strong academic performance in G1 (15) and

Student 2
-----------------------------
{
  "prediction_label": "Fail",
  "confidence_level": "Medium",
  "top_reason": "High number of absences (18) significantly impacts academic performance.",
  "second_reason": "Low study time (1 hour/day) and high failure count (3) indicate poor engagement.",
  "next_step": "Recommend increasing study time and addressing attendance issues."
}


Student 3
-----------------------------
{
  "prediction_label": "Pass",
  "confidence_level": "High",
  "top_reason": "Strong academic performance (high G1/G2 scores, good health, low failures)",
  "second_reason": "Supportive family and school environment (famsup yes, guardian mother, extracurricular activities)",
  "next_step": "Recommend continued academic support and monitoring"
}


## 13. Validate JSON Response

In [39]:
def validate_response(response):

    # Handle API failure
    if response is None:

        print("No response received from LLM.")

        return {
            "prediction_label": None,
            "confidence_level": None,
            "top_reason": None,
            "second_reason": None,
            "next_step": None
        }

    try:

        response = response.strip()

        # Extract only the JSON object
        start = response.find("{")
        end = response.rfind("}")

        if start != -1 and end != -1:
            response = response[start:end+1]

        parsed = json.loads(response)

        validate(
            instance=parsed,
            schema=schema
        )

        print("Validation Passed")

        return parsed

    except json.JSONDecodeError:

        print("JSON Parsing Failed")

    except ValidationError as e:

        print("Schema Validation Failed")

        print(e.message)

    return {
        "prediction_label": None,
        "confidence_level": None,
        "top_reason": None,
        "second_reason": None,
        "next_step": None
    }

## 14. End-to-End Demonstration

In [43]:
results = []

for i, student in enumerate(students, start=1):

    # Encode the student record
    encoded = encode_record(student)

    # Predict using the trained model
    prediction = int(best_pipeline.predict(encoded)[0])

    probability = float(best_pipeline.predict_proba(encoded)[0].max())

    # Create prompt for the LLM
    user_prompt = f"""
Student Features

{json.dumps(student, indent=2)}

Prediction : {prediction}

Prediction Probability : {probability:.3f}

Return ONLY valid JSON in this format:

{{
    "prediction_label": "Pass or Fail",
    "confidence_level": "Low, Medium or High",
    "top_reason": "string",
    "second_reason": "string",
    "next_step": "string"
}}
"""

    # Call the LLM
    raw_response = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    # Validate the response
    validated = validate_response(raw_response)

    validation_status = (
        "Pass"
        if validated["prediction_label"] is not None
        else "Fail"
    )

    # Display complete information
    print("\n===================================================")
    print(f"Student {i}")
    print("===================================================")

    print("\nFeature Input")
    print("----------------")
    print(json.dumps(student, indent=2))

    print("\nPredicted Class")
    print("----------------")
    print(prediction)

    print("\nPrediction Probability")
    print("----------------")
    print(f"{probability:.3f}")

    print("\nLLM Output")
    print("----------------")
    print(raw_response)

    print("\nValidation Status")
    print("----------------")
    print(validation_status)

    # Save results
    results.append({

        "Student": i,

        "Prediction": prediction,

        "Probability": round(probability, 3),

        "Validation": validation_status

    })

print("\nSummary Table")
display(pd.DataFrame(results))

Validation Passed

Student 1

Feature Input
----------------
{
  "school": "GP",
  "sex": "F",
  "age": 17,
  "address": "U",
  "famsize": "GT3",
  "Pstatus": "T",
  "Medu": 4,
  "Fedu": 4,
  "Mjob": "teacher",
  "Fjob": "teacher",
  "reason": "course",
  "guardian": "mother",
  "traveltime": 1,
  "studytime": 4,
  "failures": 0,
  "schoolsup": "no",
  "famsup": "yes",
  "paid": "no",
  "activities": "yes",
  "nursery": "yes",
  "higher": "yes",
  "internet": "yes",
  "romantic": "no",
  "famrel": 5,
  "freetime": 3,
  "goout": 2,
  "Dalc": 1,
  "Walc": 1,
  "health": 5,
  "absences": 2,
  "G1": 15,
  "G2": 16
}

Predicted Class
----------------
1

Prediction Probability
----------------
0.901

LLM Output
----------------
{
  "prediction_label": "Pass",
  "confidence_level": "High",
  "top_reason": "High parental education level (Medu: 4, Fedu: 4) and both parents are teachers, indicating a strong academic environment.",
  "second_reason": "Student has no prior failures (failures: 0) a

,Student,Prediction,Probability,Validation
0,1,1,0.901,Pass
1,2,0,0.739,Fail
2,3,1,0.861,Pass


## 15. Temperature Comparison

In [44]:
# Use the first student record
student = students[0]

# Encode the student record
encoded = encode_record(student)

# Predict using the trained model
prediction = int(best_pipeline.predict(encoded)[0])

probability = float(best_pipeline.predict_proba(encoded)[0].max())

# Create prompt
prompt = f"""
Student Features

{json.dumps(student, indent=2)}

Prediction : {prediction}

Prediction Probability : {probability:.3f}

Return ONLY valid JSON in this format:

{{
    "prediction_label": "Pass or Fail",
    "confidence_level": "Low, Medium or High",
    "top_reason": "string",
    "second_reason": "string",
    "next_step": "string"
}}
"""

# Temperature = 0
response_temp0 = call_llm(
    system_prompt,
    prompt,
    temperature=0
)

# Temperature = 0.7
response_temp07 = call_llm(
    system_prompt,
    prompt,
    temperature=0.7
)

# Display complete responses
print("Temperature = 0")
print("----------------")
print(response_temp0)

print("\n")

print("Temperature = 0.7")
print("----------------")
print(response_temp07)

# Comparison table
comparison = pd.DataFrame({

    "Input": [
        "Student 1"
    ],

    "Output at Temperature = 0": [
        response_temp0
    ],

    "Output at Temperature = 0.7": [
        response_temp07
    ],

    "Key Difference": [
        "Temperature 0 produced a more consistent response. Temperature 0.7 produced slightly different wording while keeping the same meaning."
    ]

})

print("\nTemperature Comparison Table")
display(comparison)

Temperature = 0
----------------
{
    "prediction_label": "Pass",
    "confidence_level": "High",
    "top_reason": "High academic performance indicated by G1 and G2 scores and low absences",
    "second_reason": "Strong family support and extracurricular activities contributing to engagement",
    "next_step": "Continue regular monitoring and encourage sustained study habits"
}


Temperature = 0.7
----------------
{"prediction_label":"Pass","confidence_level":"High","top_reason":"High studytime and good parental education levels indicate strong academic support and study habits.","second_reason":"Low absences, good health score, and no past failures suggest positive learning environment and engagement.","next_step":"Encourage the student to maintain current study routines and continue monitoring performance in upcoming exams."}

Temperature Comparison Table


,Input,Output at Temperature = 0,Output at Temperature = 0.7,Key Difference
0,Student 1,"{\n ""prediction_label"": ""Pass"",\n ""confi...","{""prediction_label"":""Pass"",""confidence_level"":...",Temperature 0 produced a more consistent respo...
